# Stage 17C: target-free selector gate

Stage 17B selectorが悪化しやすいcutを、target-free特徴だけで予測します。well-isolated 5-fold cross-fitで判定し、追加PF計算やGPUは不要です。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## Stage 17B artifact

full selector screenのcut reportとtarget-free audit特徴を使います。

In [ ]:
stage17b_run = artifact_dir / 'stage17b_selector_replay_full_v001'
assert (stage17b_run / 'cut_report.parquet').is_file(), 'Run Notebook 380 Stage 17B first'
assert (stage17b_run / 'selector_audit.parquet').is_file()
print('Stage 17B:', stage17b_run)

## Cross-fit gate

thresholdは事前固定の0.0です。診断gridは表示しますが、同じOOFを見てprimary thresholdを変更しません。

In [ ]:
RUN_ID = 'stage17c_selector_gate_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-selector-gate',
        '--config', 'configs/experiment/stage17c_selector_gate.yaml',
        '--stage17b-run', str(stage17b_run),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage17c_complete': summary['stage17c_complete'],
    'promoted_to_resolution_audit': summary['promoted_to_resolution_audit'],
    'primary_threshold': summary['primary_threshold'],
    'primary_metrics': summary['primary_metrics'],
    'bootstrap_95pct': summary['bootstrap_95pct'],
    'gain_prediction_correlation': summary['gain_prediction_correlation'],
    'threshold_grid': summary['threshold_grid'],
    'gates': summary['gates'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。gateが通過した場合だけ、stratified subsetでPF解像度・seed数・beamを監査します。